# SwRD — Brain Tumor MRI Classification (Multiclass)

---

This notebook implements the **SwRD** architecture — a parallel Swin Transformer + ResNet-50
fusion model — 

based on the research paper **"Advancing Brain Tumor MRI Classification using SwRD:
A Parallel Swin Transformer–ResNet Approach."**

The architecture and training pipeline are inspired
by the ideas presented in the paper, adapted here for experimentation and reproducibility.

The model is trained on a 31,464-image multiclass MRI corpus assembled from two Kaggle sources,
and the notebook covers the full pipeline: dataset preparation, training, evaluation, and
Grad-CAM visualisation.

**Contents:**
1. **Imports & Reproducibility** — seeds, device, dependencies
2. **Hyperparameters** — training configuration
3. **Dataset** — Hashemi + SARTAJ combined corpus
4. **Transforms & Dataset** — augmentation pipeline and PyTorch dataset
5. **Model Architecture** — SwRD (Swin-Base + ResNet-50 Fusion)
6. **Training Utilities** — `fit()`, checkpoint save / load
7. **Train or Load** — trains from scratch if no checkpoint is found
8. **Training Curves** — accuracy / loss per epoch
9. **Evaluation** — classification report + confusion matrix
10. **Load Saved Model** — reload checkpoint after kernel restart
11. **Grad-CAM** — visual explanation of model predictions


## **1. Install Dependencies**


In [1]:
%pip install -q torch torchvision timm scikit-learn albumentations matplotlib seaborn tqdm kagglehub Pillow


## **2. Imports & Reproducibility**

Set random seeds across all libraries to improve experiment reproducibility.


In [2]:
import os, random, time, math, json, copy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import ResNet50_Weights

import timm
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import albumentations as A
from albumentations.pytorch import ToTensorV2

# =============================== Reproducibility
SEED = 42

def seed_everything(seed: int = 42):
    os.environ['PYTHONHASHSEED']          = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

seed_everything(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device :', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU    :', torch.cuda.get_device_name(0))
    print('VRAM   : %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))


Device : cuda
GPU    : Tesla P100-PCIE-16GB
VRAM   : 17.1 GB


## **3. Hyperparameters**

Training hyperparameters used in this implementation:

| Hyperparameter | Value |
|:---|:---|
| Image size | 224 × 224 |
| Batch size | 32 |
| Optimizer | AdamW |
| Learning rate | 1 × 10−5 |
| Weight decay | 1 × 10−2 |
| Loss | CrossEntropyLoss |
| Train / Val split | 80 / 20 |
| Epochs | 40 |
| Early stopping patience | 12 |
| ResNet variant | ResNet-50 (pretrained) |
| Swin variant | Swin-Base patch4 window7 |


In [3]:
# =============================== Model Training Settings
IMG_SIZE     = 224
BATCH_SIZE   = 32
LR           = 1e-5
WEIGHT_DECAY = 1e-2
EPOCHS       = 40
NUM_CLASSES  = 4
NUM_WORKERS  = 4
PIN_MEMORY   = (DEVICE.type == 'cuda')

# =============================== Class Labels
CLASS_NAMES  = ['glioma', 'meningioma', 'notumor', 'pituitary']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

# =============================== ImageNet Normalisation
# Used by both ResNet-50 and Swin pretrained weights.
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)


## **4. Dataset Preparation (31,464 images)**

The dataset is built by combining two public Kaggle MRI datasets:

1. **Hashemi** — `mohammadhossein77/brain-tumors-dataset`  
   21,672 preprocessed MRI scans (brightness, noise, hist-eq, flip, rotation already applied).
2. **SARTAJ** — `sartajbhuvaji/brain-tumor-classification-mri`  
   3,264 raw scans × 3 offline augmentations (zoom / Gaussian blur / crop) = 9,792 new samples.

| Class | Hashemi | SARTAJ × 3 | **Total** |
|:---|---:|---:|---:|
| glioma | 6,307 | 2,778 | **9,085** |
| meningioma | 6,391 | 2,811 | **9,202** |
| notumor | 3,066 | 1,500 | **4,566** |
| pituitary | 5,908 | 2,703 | **8,611** |
| **Total** | 21,672 | 9,792 | **31,464** |


In [4]:
import shutil, kagglehub

DATA_ROOT = Path('./data/multiclass')

# =============================== Helper: detect stale files
def _has_foreign_files(root: Path) -> bool:
    if not root.exists(): return False
    for c in root.iterdir():
        if not c.is_dir(): continue
        for p in c.iterdir():
            if p.is_file() and not (p.name.startswith('hashemi_') or
                                     p.name.startswith('sartaj_')):
                return True
    return False

if _has_foreign_files(DATA_ROOT):
    print(f'Found non-paper files in {DATA_ROOT} -> wiping and rebuilding.')
    shutil.rmtree(DATA_ROOT)

for cls in CLASS_NAMES:
    (DATA_ROOT / cls).mkdir(parents=True, exist_ok=True)

# =============================== [1/2] Hashemi
print('[1/2] Downloading Hashemi preprocessed set...')
hashemi_path = Path(kagglehub.dataset_download('mohammadhossein77/brain-tumors-dataset'))
hashemi_map  = {
    'Data/Tumor/glioma_tumor':     'glioma',
    'Data/Tumor/meningioma_tumor': 'meningioma',
    'Data/Normal':                 'notumor',
    'Data/Tumor/pituitary_tumor':  'pituitary',
}
h_copied = 0
for src_rel, dst in hashemi_map.items():
    src_dir, dst_dir = hashemi_path / src_rel, DATA_ROOT / dst
    if not src_dir.exists():
        print(f'  ! missing {src_dir}'); continue
    for f in src_dir.iterdir():
        if f.suffix.lower() not in {'.jpg', '.jpeg', '.png'}: continue
        tgt = dst_dir / f'hashemi_{f.name}'
        if not tgt.exists():
            shutil.copy(f, tgt); h_copied += 1
print(f'      Hashemi images: {h_copied}')

# =============================== [2/2] SARTAJ — 3 augmentations per image
print('[2/2] Downloading SARTAJ raw set...')
sartaj_path = Path(kagglehub.dataset_download('sartajbhuvaji/brain-tumor-classification-mri'))
sartaj_map  = {
    'Training/glioma_tumor':     'glioma',
    'Training/meningioma_tumor': 'meningioma',
    'Training/no_tumor':         'notumor',
    'Training/pituitary_tumor':  'pituitary',
    'Testing/glioma_tumor':      'glioma',
    'Testing/meningioma_tumor':  'meningioma',
    'Testing/no_tumor':          'notumor',
    'Testing/pituitary_tumor':   'pituitary',
}

aug_zoom = A.Compose([
    A.Resize(248, 248),
    A.CenterCrop(224, 224),
    A.Affine(scale=(1.15, 1.25), p=1.0),
])
aug_blur = A.Compose([
    A.Resize(224, 224),
    A.GaussianBlur(blur_limit=(5, 7), sigma_limit=(1.0, 1.5), p=1.0),
])
aug_crop = A.Compose([
    A.Resize(268, 268),
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.8, 1.0), p=1.0),
])
sartaj_augs = [('zoom', aug_zoom), ('blur', aug_blur), ('crop', aug_crop)]

s_copied = 0
for src_rel, dst in sartaj_map.items():
    src_dir = sartaj_path / src_rel
    dst_dir = DATA_ROOT / dst
    if not src_dir.exists(): continue
    for f in src_dir.iterdir():
        if f.suffix.lower() not in {'.jpg', '.jpeg', '.png'}: continue
        img = np.array(Image.open(f).convert('RGB'))
        for aug_name, aug in sartaj_augs:
            tgt = dst_dir / f'sartaj_{aug_name}_{f.name}'
            if not tgt.exists():
                aug_img = aug(image=img)['image']
                Image.fromarray(aug_img).save(tgt)
                s_copied += 1
print(f'      SARTAJ augmented images placed: {s_copied}')

# =============================== Build DataFrame
records = []
for cls in CLASS_NAMES:
    for p in (DATA_ROOT / cls).iterdir():
        if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
            records.append({'path': str(p), 'class': cls, 'label': CLASS_TO_IDX[cls]})

df = pd.DataFrame(records)
print(f'\nTotal images: {len(df)}')
print(df['class'].value_counts())


[1/2] Downloading Hashemi preprocessed set...
      Hashemi images: 21672
[2/2] Downloading SARTAJ raw set...
      SARTAJ augmented images placed: 9792

Total images: 31464
class
meningioma    9202
glioma        9085
pituitary     8611
notumor       4566
Name: count, dtype: int64


## **5. Transforms & PyTorch Dataset**

Training uses three extra augmentations (zoom / blur / crop) on top of ImageNet normalisation.
Validation only resizes and normalises.


In [5]:
# =============================== Training Pipeline
train_tf = A.Compose([
    A.Resize(int(IMG_SIZE * 1.15), int(IMG_SIZE * 1.15)),
    A.RandomCrop(IMG_SIZE, IMG_SIZE, p=1.0),          # cropping
    A.Affine(scale=(0.85, 1.15), p=0.5),              # zoom
    A.GaussianBlur(blur_limit=(3, 5), p=0.3),         # Gaussian blur
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.4),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

# =============================== Validation Pipeline
val_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])


class BrainMRIDataset(Dataset):
    def __init__(self, df, transform):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = np.array(Image.open(row['path']).convert('RGB'))
        img = self.transform(image=img)['image']
        return img, int(row['label'])


# =============================== Stratified 80 / 20 Split
train_df, val_df = train_test_split(
    df, test_size=0.20, stratify=df['label'], random_state=SEED
)

print('Train:', len(train_df), '| Val:', len(val_df))
print('\nTrain per class:'); print(train_df['class'].value_counts())
print('\nVal per class:');   print(val_df['class'].value_counts())

train_ds = BrainMRIDataset(train_df, train_tf)
val_ds   = BrainMRIDataset(val_df,   val_tf)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)


Train: 25171 | Val: 6293

Train per class:
class
meningioma    7362
glioma        7268
pituitary     6889
notumor       3652
Name: count, dtype: int64

Val per class:
class
meningioma    1840
glioma        1817
pituitary     1722
notumor        914
Name: count, dtype: int64


## **6. Model Architecture — SwRD**

Parallel ResNet-50 + Swin-Base branches. Both receive the same
`(B, 3, 224, 224)` input tensor and their pooled features are
concatenated before the final linear classifier.

| Branch | Backbone | Output dim |
|:---|:---|:---|
| ResNet-50 | ImageNet pretrained, drop final FC | 2048 |
| Swin-Base | patch4 window7 224, drop head | 1024 |
| Fusion | `torch.cat` | 3072 |
| Classifier | `Linear(3072, 4)` | 4 |


In [6]:
# =============================== SwRD Model
class SwRD(nn.Module):
    """Parallel Swin-Transformer + ResNet-50 with plain concatenation fusion."""

    def __init__(self, num_classes: int = 4, pretrained: bool = True):
        super().__init__()

# =============================== ResNet-50 Branch
        weights = ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        resnet  = models.resnet50(weights=weights)
        self.resnet_features = nn.Sequential(*list(resnet.children())[:-1])
        self.resnet_out_dim  = resnet.fc.in_features   # 2048

# =============================== Swin-Base Branch
        self.swin = timm.create_model(
            'swin_base_patch4_window7_224',
            pretrained=pretrained,
            num_classes=0,
            global_pool='avg',
        )
        self.swin_out_dim = self.swin.num_features     # 1024

# =============================== Classifier Head
        self.classifier = nn.Linear(
            self.resnet_out_dim + self.swin_out_dim,   # 3072
            num_classes
        )

    def forward(self, x):
        f_res  = self.resnet_features(x).flatten(1)   # (B, 2048)
        f_swin = self.swin(x)                         # (B, 1024)
        f_cat  = torch.cat([f_res, f_swin], dim=1)   # (B, 3072)
        return self.classifier(f_cat)                 # (B, num_classes)


CHECKPOINT_PATH = 'swrd_multiclass_best.pt'

# Quick param count + shape check
_m = SwRD(num_classes=NUM_CLASSES, pretrained=False)
print(f'SwRD  params = {sum(p.numel() for p in _m.parameters())/1e6:.2f} M')
with torch.no_grad():
    _out = _m(torch.randn(2, 3, IMG_SIZE, IMG_SIZE))
print(f'Shape check  : input (2, 3, {IMG_SIZE}, {IMG_SIZE}) -> output {tuple(_out.shape)}')
del _m


SwRD  params = 108.25 M
Shape check  : input (2, 3, 224, 224) -> output (2, 4)


## **7. Training Utilities**

| Component | Choice | Reason |
|:----------|:-------|:-------|
| **Optimizer** | `AdamW` (lr = `1e-5`, wd = `1e-2`) | Stable convergence for transformer backbones |
| **Loss** | `CrossEntropyLoss` | Multiclass classification |
| **Checkpoints** | Full bundle (weights + history + metadata) | Resume + reproducibility |
| **Early stopping** | patience = 12, monitor = val_acc | Prevent overfitting |


In [7]:
def run_epoch(model, loader, criterion, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    total_loss, total_correct, total = 0.0, 0, 0

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in tqdm(loader, leave=False, desc='train' if train else 'val'):
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            if train:
                optimizer.zero_grad(set_to_none=True)

            logits = model(x)
            loss   = criterion(logits, y)

            if train:
                loss.backward()
                optimizer.step()

            total_loss    += loss.item() * x.size(0)
            total_correct += (logits.argmax(1) == y).sum().item()
            total         += x.size(0)

    return total_loss / total, total_correct / total


def save_checkpoint_bundle(path, model, history=None, best_acc=None, extra=None):
    torch.save({
        'model_state_dict': model.state_dict(),
        'history':          history  if history  is not None else {},
        'best_acc':         float(best_acc) if best_acc is not None else None,
        'extra':            extra    if extra    is not None else {},
    }, str(path))


def load_checkpoint_bundle(path, map_location=None):
    ckpt = torch.load(str(path), map_location=map_location)
    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
        return ckpt
    # Backward compatibility: plain state_dict
    return {
        'model_state_dict': ckpt,
        'history': {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []},
        'best_acc': None,
        'extra': {},
    }


def fit(model, train_loader, val_loader, epochs=EPOCHS, lr=LR, wd=WEIGHT_DECAY,
        early_stopping_patience=12):
    """Train `model` for `epochs`, restore best weights, return (history, best_acc, best_state)."""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_acc, best_state, no_improve = 0.0, None, 0

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
        va_loss, va_acc = run_epoch(model, val_loader,   criterion)

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss)
        history['val_acc'].append(va_acc)

        flag = ''
        if va_acc > best_acc:
            best_acc   = va_acc
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
            flag = '  ✓ best'
        else:
            no_improve += 1

        print(f'Epoch {epoch:>2}/{epochs}  '
              f'train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f}  '
              f'val_loss={va_loss:.4f}  val_acc={va_acc:.4f}  '
              f'({time.time()-t0:.0f}s){flag}')

        if no_improve >= early_stopping_patience:
            print(f'  Early stopping at epoch {epoch} (patience={early_stopping_patience})')
            break

    return history, best_acc, best_state


## **8. Train or Load**

- **Checkpoint found** → load weights and skip training
- **No checkpoint** → train from scratch, save checkpoint

> Set `FORCE_TRAIN = True` to retrain even if a checkpoint already exists.


In [8]:
FORCE_TRAIN = False   # True -> retrain even if checkpoint exists

if os.path.exists(CHECKPOINT_PATH) and not FORCE_TRAIN:
# =============================== Load from checkpoint
    try:
        ckpt  = load_checkpoint_bundle(CHECKPOINT_PATH, map_location=DEVICE)
        model = SwRD(num_classes=NUM_CLASSES, pretrained=False).to(DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        model.eval()
        history  = ckpt.get('history', {})
        best_acc = ckpt.get('best_acc')
        acc_str  = f'{best_acc*100:.2f}%' if best_acc else 'N/A'
        print(f'Loaded checkpoint : {CHECKPOINT_PATH}')
        print(f'Best val accuracy : {acc_str}')
    except Exception as e:
        print(f'Load failed: {e}  -> training from scratch...')
        os.remove(CHECKPOINT_PATH)
        FORCE_TRAIN = True

if not os.path.exists(CHECKPOINT_PATH) or FORCE_TRAIN:
# =============================== Train from scratch
    reason = 'FORCE_TRAIN=True' if FORCE_TRAIN else 'no checkpoint found'
    print(f'Training SwRD from scratch ({reason})...')
    model = SwRD(num_classes=NUM_CLASSES, pretrained=True).to(DEVICE)
    history, best_acc, best_state = fit(model, train_loader, val_loader)
    model.load_state_dict(best_state)
    model.eval()
    save_checkpoint_bundle(
        CHECKPOINT_PATH, model,
        history=history, best_acc=best_acc,
        extra={'epochs': EPOCHS, 'lr': LR, 'wd': WEIGHT_DECAY},
    )
    print(f'Saved checkpoint  : {CHECKPOINT_PATH}')
    print(f'Best val accuracy : {best_acc*100:.2f}%')


Training SwRD from scratch (no checkpoint found)...
Epoch  1/40  train_loss=0.3254  train_acc=0.8763  val_loss=0.1022  val_acc=0.9663  (170s)  ✓ best
Epoch  2/40  train_loss=0.0720  train_acc=0.9781  val_loss=0.0614  val_acc=0.9825  (171s)  ✓ best
Epoch  3/40  train_loss=0.0442  train_acc=0.9865  val_loss=0.0467  val_acc=0.9862  (171s)  ✓ best
Epoch  4/40  train_loss=0.0315  train_acc=0.9898  val_loss=0.0412  val_acc=0.9884  (172s)  ✓ best
Epoch  5/40  train_loss=0.0249  train_acc=0.9924  val_loss=0.0371  val_acc=0.9878  (171s)
Epoch  6/40  train_loss=0.0200  train_acc=0.9936  val_loss=0.0329  val_acc=0.9894  (172s)  ✓ best
Epoch  7/40  train_loss=0.0169  train_acc=0.9948  val_loss=0.0323  val_acc=0.9898  (171s)  ✓ best
Epoch  8/40  train_loss=0.0145  train_acc=0.9956  val_loss=0.0323  val_acc=0.9894  (170s)
Epoch  9/40  train_loss=0.0121  train_acc=0.9963  val_loss=0.0291  val_acc=0.9908  (171s)  ✓ best
Epoch 10/40  train_loss=0.0109  train_acc=0.9967  val_loss=0.0305  val_acc=0.9906 

## **9. Training Curves**

Plots accuracy and loss per epoch — solid line for validation, dashed for training.


In [9]:
# =============================== Training History Visualization
if not history or not history.get('train_loss'):
    print('No history available. Run Section 8 first.')
else:
    ep = range(1, len(history['train_loss']) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle('SwRD — Training Curves', fontsize=14, fontweight='bold')

    # =============================== Accuracy
    axes[0].plot(ep, history['val_acc'],   color='#2E86AB', lw=2,   label='Val Accuracy')
    axes[0].plot(ep, history['train_acc'], color='#2E86AB', lw=1.2, ls='--', alpha=0.5, label='Train Accuracy')
    axes[0].set_title('Accuracy', fontsize=12)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(alpha=0.25)

    # =============================== Loss
    axes[1].plot(ep, history['val_loss'],   color='#E63946', lw=2,   label='Val Loss')
    axes[1].plot(ep, history['train_loss'], color='#E63946', lw=1.2, ls='--', alpha=0.5, label='Train Loss')
    axes[1].set_title('Loss', fontsize=12)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(alpha=0.25)

    plt.tight_layout()
    plt.show()

    best_ep = int(np.argmin(history['val_loss'])) + 1
    print(f'Best epoch (lowest val_loss) : {best_ep}')
    print(f'Best val accuracy            : {max(history["val_acc"]):.4f}')
    print(f'Best val loss                : {min(history["val_loss"]):.4f}')


Best epoch (lowest val_loss) : 27
Best val accuracy            : 0.9919
Best val loss                : 0.0264


<Figure size 1300x400 with 2 Axes>

## **10. Evaluation — Classification Report & Confusion Matrix**

Runs inference on the validation set and prints a full per-class report
followed by a confusion matrix.


In [10]:
def predict_all(model, loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in tqdm(loader, desc='predict', leave=False):
            x = x.to(DEVICE, non_blocking=True)
            ps.append(model(x).argmax(1).cpu().numpy())
            ys.append(y.numpy())
    return np.concatenate(ys), np.concatenate(ps)


y_true, y_pred = predict_all(model, val_loader)

# =============================== Classification Report
print('Classification Report — SwRD')
print('-' * 55)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

# =============================== Confusion Matrix
cm_data = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm_data, annot=True, fmt='d',
    cmap=sns.light_palette('#2E86AB', as_cmap=True),
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    linewidths=0.5, ax=ax,
)
ax.set_title('Confusion Matrix — SwRD', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.tight_layout()
plt.show()


Classification Report — SwRD
-------------------------------------------------------
              precision    recall  f1-score   support

      glioma     0.9962    0.9928    0.9945      1817
  meningioma     0.9929    0.9935    0.9932      1840
      notumor     0.9945    0.9978    0.9962       914
   pituitary     0.9971    0.9977    0.9974      1722

    accuracy                         0.9949      6293
   macro avg     0.9952    0.9954    0.9953      6293
weighted avg     0.9950    0.9949    0.9949      6293



<Figure size 600x500 with 2 Axes>

## **11. Load Saved Model**

Reload the checkpoint after restarting the kernel — no need to retrain.


In [11]:
if not os.path.exists(CHECKPOINT_PATH):
    print(f'Checkpoint not found: {CHECKPOINT_PATH}')
    print('Run Section 8 first to train and save the model.')
else:
    ckpt     = load_checkpoint_bundle(CHECKPOINT_PATH, map_location=DEVICE)
    reloaded = SwRD(num_classes=NUM_CLASSES, pretrained=False).to(DEVICE)
    reloaded.load_state_dict(ckpt['model_state_dict'])
    reloaded.eval()

    acc_str = f"{ckpt['best_acc']*100:.2f}%" if ckpt.get('best_acc') else 'N/A'
    print(f'Loaded            : {CHECKPOINT_PATH}')
    print(f'Best val accuracy : {acc_str}')
    print(f'Extra             : {ckpt.get("extra", {})}')

    # Quick sanity check
    with torch.no_grad():
        dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
        out   = reloaded(dummy)
    print(f'Forward pass OK   : output shape = {tuple(out.shape)}')


Loaded            : swrd_multiclass_best.pt
Best val accuracy : 99.19%
Extra             : {'epochs': 40, 'lr': 1e-05, 'wd': 0.01}
Forward pass OK   : output shape = (2, 4)


## **12. Grad-CAM — Visual Explanation**

Grad-CAM highlights the image regions that drove the model's prediction.
It hooks the **last ResNet conv layer** (the spatial branch) since Swin
features are patch-pooled and lose explicit spatial layout.

- **`GRADCAM_N_IMAGES`** — how many val samples to visualise
- **`GRADCAM_ONLY_CORRECT`** — show only correctly classified images


In [12]:
# =============================== Grad-CAM Configuration
GRADCAM_N_IMAGES     = 8      # number of samples to visualise
GRADCAM_ONLY_CORRECT = True   # True -> only show correct predictions


# =============================== Grad-CAM Implementation
class GradCAM:
    """Hooks the last ResNet conv block and computes class activation maps."""

    def __init__(self, model):
        self.model       = model
        self.gradients   = None
        self.activations = None
        self._hooks      = []

        # Target: last conv layer of ResNet branch (layer4[-1])
        target_layer = list(model.resnet_features.children())[-2][-1]

        self._hooks.append(target_layer.register_forward_hook(self._save_activation))
        self._hooks.append(target_layer.register_full_backward_hook(self._save_gradient))

    def _save_activation(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, x, class_idx=None):
        """Returns (pred_idx, cam_np) where cam_np is (H, W) in [0, 1]."""
        self.model.zero_grad()
        logits = self.model(x)
        pred   = logits.argmax(1).item() if class_idx is None else class_idx
        logits[0, pred].backward()

        weights = self.gradients.mean(dim=[2, 3], keepdim=True)   # (1, C, 1, 1)
        cam     = (weights * self.activations).sum(dim=1).squeeze()  # (H, W)
        cam     = F.relu(cam)
        cam    -= cam.min()
        if cam.max() > 0:
            cam /= cam.max()

        return pred, cam.cpu().numpy()

    def remove_hooks(self):
        for h in self._hooks:
            h.remove()


# =============================== Denormalize helper
def denorm(t):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (t.cpu() * std + mean).clamp(0, 1)


# =============================== Run Grad-CAM
gcam = GradCAM(model)
model.eval()

collected = []
for x_batch, y_batch in val_loader:
    for i in range(x_batch.size(0)):
        if len(collected) >= GRADCAM_N_IMAGES:
            break
        x_single = x_batch[i:i+1].to(DEVICE)
        label     = y_batch[i].item()
        pred_idx, heatmap = gcam.generate(x_single)
        if GRADCAM_ONLY_CORRECT and pred_idx != label:
            continue
        collected.append((x_single.squeeze(0).cpu(), label, pred_idx, heatmap))
    if len(collected) >= GRADCAM_N_IMAGES:
        break

gcam.remove_hooks()

# =============================== Grad-CAM Visualization
ncols = 4
nrows = math.ceil(len(collected) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 3.5))
axes = np.array(axes).reshape(-1)

for ax in axes:
    ax.axis('off')

for idx, (img_t, label, pred_idx, cam) in enumerate(collected):
    img_np = denorm(img_t).permute(1, 2, 0).numpy()

    cam_resized = np.array(
        Image.fromarray((cam * 255).astype(np.uint8)).resize(
            (IMG_SIZE, IMG_SIZE), Image.BILINEAR
        )
    ) / 255.0

    hmap    = cm.get_cmap('RdYlGn_r')(cam_resized)[..., :3]
    overlay = (0.55 * img_np + 0.45 * hmap).clip(0, 1)

    axes[idx].imshow(overlay)
    status = '✓' if pred_idx == label else '✗'
    axes[idx].set_title(
        f'{status} True: {CLASS_NAMES[label]}\nPred: {CLASS_NAMES[pred_idx]}',
        fontsize=9,
        color='#2A5C3F' if pred_idx == label else '#8B0000',
    )
    axes[idx].axis('off')

fig.suptitle(
    'Grad-CAM — SwRD\n(warm regions = high activation)',
    fontsize=13, fontweight='bold', y=1.01,
)
plt.tight_layout()
plt.show()
print(f'Displayed {len(collected)} Grad-CAM maps.')


Displayed 8 Grad-CAM maps.


<Figure size 1400x700 with 16 Axes>